# Phase 6 — Performance Evaluation and Comparison

This Google Colab notebook evaluates the trained **PPO crop-planning agent** from Phase 4 and compares it with a **random crop-selection baseline**.

## Required uploads

1. `proceed_data.csv` — the same processed dataset used during PPO training.
2. `Farmer_RL_Phase4_Complete.zip` — the ZIP downloaded from the Phase 4 notebook.

## Main outputs

- PPO episode results
- Random-policy episode results
- PPO versus random comparison table
- Statistical summary
- Reward, savings, income, survival and crop-frequency graphs
- `Farmer_RL_Phase6_Results.zip`

> This notebook does not retrain PPO.

In [ ]:
# Install maintained reinforcement-learning libraries.
!pip -q install "gymnasium>=1.0,<2.0" "stable-baselines3>=2.4,<3.0" pandas matplotlib

In [ ]:
import json
import shutil
import warnings
import zipfile
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gymnasium import spaces
from IPython.display import display
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

warnings.filterwarnings("ignore", category=DeprecationWarning)

SEED = 42
N_EVAL_EPISODES = 100
HORIZON_YEARS = 20
INITIAL_SAVINGS = 100_000.0

Path("phase4_files").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
Path("graphs").mkdir(exist_ok=True)

print("Libraries imported successfully.")

## 1. Upload and prepare the processed dataset

In [ ]:
from google.colab import files

print("Upload proceed_data.csv — use the same CSV used in Phase 4.")
uploaded_csv = files.upload()

csv_names = [
    name for name in uploaded_csv
    if name.lower().endswith(".csv")
]

if not csv_names:
    raise FileNotFoundError("No CSV file was uploaded.")

CSV_PATH = csv_names[0]
print("Dataset selected:", CSV_PATH)

In [ ]:
REQUIRED_COLUMNS = [
    "Crop",
    "Annual_Rainfall",
    "Yield",
    "Modal_Price",
    "Profit",
]

OPTIONAL_NUMERIC_COLUMNS = [
    "Crop_Year",
    "Area",
    "Production",
    "Fertilizer",
    "Pesticide",
    "Revenue",
    "Cost",
]

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.astype(str).str.strip()

missing = [column for column in REQUIRED_COLUMNS if column not in df.columns]
if missing:
    raise ValueError(
        "Dataset is missing required columns: " + ", ".join(missing)
    )

df["Crop"] = df["Crop"].astype(str).str.strip()
df = df[
    df["Crop"].notna()
    & df["Crop"].ne("")
    & df["Crop"].str.lower().ne("nan")
].copy()

numeric_columns = [
    column
    for column in REQUIRED_COLUMNS[1:] + OPTIONAL_NUMERIC_COLUMNS
    if column in df.columns
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in REQUIRED_COLUMNS[1:]:
    median_value = df[column].median()
    if pd.isna(median_value):
        raise ValueError(
            f"Column {column!r} does not contain usable numeric values."
        )
    df[column] = df[column].fillna(median_value)

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=REQUIRED_COLUMNS).reset_index(drop=True)

if len(df) == 0:
    raise ValueError("No valid dataset rows remain after cleaning.")

dataset_crops = sorted(df["Crop"].unique().tolist())

print("Dataset rows:", len(df))
print("Number of crops:", len(dataset_crops))
print("Crop order:", dataset_crops)
display(df.head())

## 2. Upload and extract Phase 4 outputs

In [ ]:
from google.colab import files

print("Upload Farmer_RL_Phase4_Complete.zip.")
uploaded_zip = files.upload()

zip_names = [
    name for name in uploaded_zip
    if name.lower().endswith(".zip")
]

if not zip_names:
    raise FileNotFoundError("No ZIP file was uploaded.")

PHASE4_ZIP = zip_names[0]

extract_dir = Path("phase4_files")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)

with zipfile.ZipFile(PHASE4_ZIP, "r") as archive:
    archive.extractall(extract_dir)

print("Extracted:", PHASE4_ZIP)

In [ ]:
def find_first(root: Path, names):
    for name in names:
        matches = list(root.rglob(name))
        if matches:
            return matches[0]
    return None


BEST_MODEL_PATH = find_first(
    Path("phase4_files"),
    ["best_model.zip"],
)

FINAL_MODEL_PATH = find_first(
    Path("phase4_files"),
    ["ppo_farmer_model.zip", "ppo_farmer_model"],
)

VEC_NORMALIZE_PATH = find_first(
    Path("phase4_files"),
    ["vec_normalize.pkl"],
)

TRAINING_CONFIG_PATH = find_first(
    Path("phase4_files"),
    ["training_config.json"],
)

MODEL_PATH = BEST_MODEL_PATH or FINAL_MODEL_PATH

if MODEL_PATH is None:
    raise FileNotFoundError(
        "Could not find best_model.zip or ppo_farmer_model.zip "
        "inside the Phase 4 package."
    )

if VEC_NORMALIZE_PATH is None:
    raise FileNotFoundError(
        "Could not find vec_normalize.pkl inside the Phase 4 package."
    )

training_config = {}
if TRAINING_CONFIG_PATH is not None:
    training_config = json.loads(
        TRAINING_CONFIG_PATH.read_text(encoding="utf-8")
    )

trained_crops = training_config.get("crops", dataset_crops)

if list(trained_crops) != list(dataset_crops):
    raise ValueError(
        "Crop order differs from Phase 4. "
        "Use exactly the same proceed_data.csv used for training.\n"
        f"Training crops: {trained_crops}\n"
        f"Current crops: {dataset_crops}"
    )

HORIZON_YEARS = int(
    training_config.get("horizon_years", HORIZON_YEARS)
)
INITIAL_SAVINGS = float(
    training_config.get("initial_savings", INITIAL_SAVINGS)
)

print("Model:", MODEL_PATH)
print("VecNormalize:", VEC_NORMALIZE_PATH)
print("Planning horizon:", HORIZON_YEARS)
print("Initial savings:", INITIAL_SAVINGS)

## 3. Recreate the Phase 4 FarmEnv

In [ ]:
class FarmEnv(gym.Env):
    """Crop-selection environment used during Phase 4 training."""

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        data: pd.DataFrame,
        horizon_years: int = 20,
        initial_savings: float = 100_000.0,
        annual_living_cost: float | None = None,
        rainfall_volatility: float = 0.12,
        price_volatility: float = 0.10,
        rotation_penalty_rate: float = 0.20,
        reward_scale: float | None = None,
        render_mode: str | None = None,
    ):
        super().__init__()

        self.data = data.copy().reset_index(drop=True)

        required = {
            "Crop",
            "Annual_Rainfall",
            "Yield",
            "Modal_Price",
            "Profit",
        }
        missing = required.difference(self.data.columns)
        if missing:
            raise ValueError(
                f"FarmEnv missing columns: {sorted(missing)}"
            )

        self.crops = sorted(
            self.data["Crop"].astype(str).unique().tolist()
        )
        if len(self.crops) < 2:
            raise ValueError(
                "FarmEnv requires at least two crops."
            )

        self.crop_rows = {
            crop: self.data.index[
                self.data["Crop"] == crop
            ].to_numpy()
            for crop in self.crops
        }

        self.horizon_years = int(horizon_years)
        self.initial_savings = float(initial_savings)
        self.rainfall_volatility = float(rainfall_volatility)
        self.price_volatility = float(price_volatility)
        self.rotation_penalty_rate = float(rotation_penalty_rate)
        self.render_mode = render_mode

        typical_profit = float(
            np.nanmedian(
                np.abs(self.data["Profit"].to_numpy())
            )
        )
        self.profit_scale = max(typical_profit, 1.0)
        self.reward_scale = float(
            reward_scale or self.profit_scale
        )
        self.annual_living_cost = float(
            annual_living_cost
            if annual_living_cost is not None
            else 0.15 * self.profit_scale
        )
        self.savings_scale = max(
            abs(self.initial_savings)
            + self.horizon_years * self.profit_scale,
            1.0,
        )

        self.rain_min, self.rain_max = self._safe_bounds(
            "Annual_Rainfall"
        )
        self.price_min, self.price_max = self._safe_bounds(
            "Modal_Price"
        )
        self.yield_min, self.yield_max = self._safe_bounds(
            "Yield"
        )

        self.action_space = spaces.Discrete(len(self.crops))
        self.observation_space = spaces.Box(
            low=np.full(6, -1.0, dtype=np.float32),
            high=np.full(6, 1.0, dtype=np.float32),
            dtype=np.float32,
        )

        self.current_year = 0
        self.savings = self.initial_savings
        self.last_crop_index = -1
        self.current_context = None
        self.history = []

    def _safe_bounds(self, column: str):
        values = self.data[column].astype(float).to_numpy()
        low = float(np.nanmin(values))
        high = float(np.nanmax(values))

        if not np.isfinite(low) or not np.isfinite(high):
            return 0.0, 1.0

        if np.isclose(low, high):
            high = low + 1.0

        return low, high

    @staticmethod
    def _scale(value: float, low: float, high: float):
        scaled = (
            2.0 * (float(value) - low) / (high - low) - 1.0
        )
        return float(np.clip(scaled, -1.0, 1.0))

    def _sample_context(self):
        row = self.data.iloc[
            int(self.np_random.integers(0, len(self.data)))
        ]
        return {
            "rainfall": float(row["Annual_Rainfall"]),
            "price": float(row["Modal_Price"]),
            "yield": float(row["Yield"]),
        }

    def _get_observation(self):
        context = self.current_context

        progress = (
            2.0
            * (
                self.current_year
                / max(self.horizon_years, 1)
            )
            - 1.0
        )

        savings_scaled = np.clip(
            self.savings / self.savings_scale,
            -1.0,
            1.0,
        )

        last_crop_scaled = (
            -1.0
            if self.last_crop_index < 0
            else self._scale(
                self.last_crop_index,
                0,
                max(len(self.crops) - 1, 1),
            )
        )

        return np.array(
            [
                progress,
                savings_scaled,
                self._scale(
                    context["rainfall"],
                    self.rain_min,
                    self.rain_max,
                ),
                self._scale(
                    context["price"],
                    self.price_min,
                    self.price_max,
                ),
                self._scale(
                    context["yield"],
                    self.yield_min,
                    self.yield_max,
                ),
                last_crop_scaled,
            ],
            dtype=np.float32,
        )

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)

        self.current_year = 0
        self.savings = self.initial_savings
        self.last_crop_index = -1
        self.current_context = self._sample_context()
        self.history = []

        observation = self._get_observation()
        info = {
            "year": self.current_year,
            "savings": self.savings,
            "crops": self.crops,
        }
        return observation, info

    def step(self, action):
        action = int(action)

        if not self.action_space.contains(action):
            raise ValueError(
                f"Invalid action {action}; expected "
                f"0-{len(self.crops) - 1}."
            )

        crop = self.crops[action]
        candidate_indices = self.crop_rows[crop]
        row_index = int(
            self.np_random.choice(candidate_indices)
        )
        row = self.data.iloc[row_index]

        base_profit = float(row["Profit"])
        crop_rainfall = max(
            float(row["Annual_Rainfall"]),
            1e-6,
        )
        crop_price = max(
            float(row["Modal_Price"]),
            1e-6,
        )

        rain_noise = float(
            self.np_random.normal(
                1.0,
                self.rainfall_volatility,
            )
        )
        price_noise = float(
            self.np_random.normal(
                1.0,
                self.price_volatility,
            )
        )

        rainfall_ratio = (
            self.current_context["rainfall"] * rain_noise
        ) / crop_rainfall

        price_ratio = (
            self.current_context["price"] * price_noise
        ) / crop_price

        climate_factor = float(
            np.clip(
                0.5 + 0.5 * rainfall_ratio,
                0.35,
                1.35,
            )
        )
        market_factor = float(
            np.clip(price_ratio, 0.50, 1.50)
        )

        realised_profit = (
            base_profit
            * climate_factor
            * market_factor
        )

        rotation_penalty = 0.0
        if action == self.last_crop_index:
            rotation_penalty = (
                abs(realised_profit)
                * self.rotation_penalty_rate
            )

        net_income = (
            realised_profit
            - rotation_penalty
            - self.annual_living_cost
        )

        self.savings += net_income
        self.current_year += 1

        bankruptcy_penalty = 0.0
        terminated = self.savings <= 0.0

        if terminated:
            bankruptcy_penalty = self.profit_scale

        truncated = self.current_year >= self.horizon_years

        reward = (
            net_income - bankruptcy_penalty
        ) / self.reward_scale

        record = {
            "year": self.current_year,
            "crop": crop,
            "base_profit": base_profit,
            "realised_profit": realised_profit,
            "rotation_penalty": rotation_penalty,
            "living_cost": self.annual_living_cost,
            "net_income": net_income,
            "savings": self.savings,
            "reward": reward,
        }

        self.history.append(record)
        self.last_crop_index = action

        if not (terminated or truncated):
            self.current_context = self._sample_context()

        observation = self._get_observation()
        info = record.copy()

        return (
            observation,
            float(reward),
            bool(terminated),
            bool(truncated),
            info,
        )

    def get_history(self):
        return pd.DataFrame(self.history)

## 4. Load the trained PPO model and normalization statistics

In [ ]:
def make_monitored_env(seed_value):
    def _init():
        env = FarmEnv(
            df,
            horizon_years=HORIZON_YEARS,
            initial_savings=INITIAL_SAVINGS,
        )
        env = Monitor(env)
        env.reset(seed=seed_value)
        return env

    return _init


raw_ppo_vec = DummyVecEnv(
    [make_monitored_env(SEED + 10_000)]
)

ppo_vec = VecNormalize.load(
    str(VEC_NORMALIZE_PATH),
    raw_ppo_vec,
)

ppo_vec.training = False
ppo_vec.norm_reward = False

ppo_model = PPO.load(
    str(MODEL_PATH),
    env=ppo_vec,
)

if ppo_model.action_space.n != len(dataset_crops):
    raise ValueError(
        "Model action-space size does not match the "
        "number of crops in the uploaded dataset."
    )

print("PPO model loaded successfully.")
print("Model actions:", ppo_model.action_space.n)

## 5. Evaluation method

Both policies are evaluated for the same number of episodes.

- PPO uses the trained policy with deterministic actions.
- Random uses uniformly random crop actions.
- Episode seeds are matched so both methods face comparable stochastic conditions.
- Metrics use the environment's original, unnormalized reward.

In [ ]:
def summarize_episode(
    policy_name,
    episode_number,
    episode_reward,
    records,
):
    history = pd.DataFrame(records)

    if history.empty:
        raise RuntimeError(
            f"{policy_name} episode {episode_number} "
            "produced no records."
        )

    final_savings = float(
        history["savings"].iloc[-1]
    )

    return {
        "policy": policy_name,
        "episode": episode_number,
        "total_reward": float(episode_reward),
        "final_savings": final_savings,
        "total_net_income": float(
            history["net_income"].sum()
        ),
        "average_annual_income": float(
            history["net_income"].mean()
        ),
        "years_survived": int(len(history)),
        "bankrupt": bool(final_savings <= 0.0),
        "unique_crops_selected": int(
            history["crop"].nunique()
        ),
    }


def evaluate_ppo(
    model,
    vec_env,
    n_episodes,
):
    summaries = []
    yearly_records = []

    for episode in range(1, n_episodes + 1):
        episode_seed = SEED + episode

        # Seed the underlying environment before vector reset.
        base_env = vec_env.venv.envs[0].unwrapped
        base_env.reset(seed=episode_seed)

        obs = vec_env.reset()
        done = np.array([False])
        episode_reward = 0.0
        records = []

        while not bool(done[0]):
            action, _ = model.predict(
                obs,
                deterministic=True,
            )

            obs, reward, done, infos = vec_env.step(
                action
            )

            step_info = infos[0]
            episode_reward += float(reward[0])

            if "crop" in step_info:
                record = {
                    key: step_info.get(key)
                    for key in [
                        "year",
                        "crop",
                        "base_profit",
                        "realised_profit",
                        "rotation_penalty",
                        "living_cost",
                        "net_income",
                        "savings",
                        "reward",
                    ]
                }
                record["policy"] = "PPO"
                record["episode"] = episode
                records.append(record)

        summaries.append(
            summarize_episode(
                "PPO",
                episode,
                episode_reward,
                records,
            )
        )
        yearly_records.extend(records)

    return (
        pd.DataFrame(summaries),
        pd.DataFrame(yearly_records),
    )


def evaluate_random(
    data,
    n_episodes,
):
    summaries = []
    yearly_records = []

    for episode in range(1, n_episodes + 1):
        episode_seed = SEED + episode

        env = FarmEnv(
            data,
            horizon_years=HORIZON_YEARS,
            initial_savings=INITIAL_SAVINGS,
        )

        observation, info = env.reset(
            seed=episode_seed
        )

        random_generator = np.random.default_rng(
            episode_seed + 50_000
        )

        terminated = False
        truncated = False
        episode_reward = 0.0
        records = []

        while not (terminated or truncated):
            action = int(
                random_generator.integers(
                    0,
                    env.action_space.n,
                )
            )

            (
                observation,
                reward,
                terminated,
                truncated,
                step_info,
            ) = env.step(action)

            episode_reward += float(reward)

            record = {
                key: step_info.get(key)
                for key in [
                    "year",
                    "crop",
                    "base_profit",
                    "realised_profit",
                    "rotation_penalty",
                    "living_cost",
                    "net_income",
                    "savings",
                    "reward",
                ]
            }
            record["policy"] = "Random"
            record["episode"] = episode
            records.append(record)

        summaries.append(
            summarize_episode(
                "Random",
                episode,
                episode_reward,
                records,
            )
        )
        yearly_records.extend(records)

        env.close()

    return (
        pd.DataFrame(summaries),
        pd.DataFrame(yearly_records),
    )

## 6. Run PPO and random-policy evaluation

In [ ]:
print(
    f"Evaluating PPO for {N_EVAL_EPISODES} episodes..."
)
ppo_results, ppo_yearly = evaluate_ppo(
    ppo_model,
    ppo_vec,
    N_EVAL_EPISODES,
)

print(
    f"Evaluating random baseline for "
    f"{N_EVAL_EPISODES} episodes..."
)
random_results, random_yearly = evaluate_random(
    df,
    N_EVAL_EPISODES,
)

all_results = pd.concat(
    [ppo_results, random_results],
    ignore_index=True,
)

all_yearly = pd.concat(
    [ppo_yearly, random_yearly],
    ignore_index=True,
)

print("Evaluation completed.")
display(all_results.head())

## 7. Build the comparison tables

In [ ]:
metric_columns = [
    "total_reward",
    "final_savings",
    "total_net_income",
    "average_annual_income",
    "years_survived",
    "bankrupt",
    "unique_crops_selected",
]

comparison_rows = []

for policy_name, policy_df in all_results.groupby(
    "policy"
):
    row = {"policy": policy_name}

    for metric in metric_columns:
        numeric_values = pd.to_numeric(
            policy_df[metric],
            errors="coerce",
        )
        row[f"{metric}_mean"] = float(
            numeric_values.mean()
        )
        row[f"{metric}_std"] = float(
            numeric_values.std(ddof=1)
        )

    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

ppo_summary = comparison_df[
    comparison_df["policy"] == "PPO"
].iloc[0]

random_summary = comparison_df[
    comparison_df["policy"] == "Random"
].iloc[0]

improvement_rows = []

for metric in [
    "total_reward",
    "final_savings",
    "total_net_income",
    "average_annual_income",
    "years_survived",
]:
    ppo_value = float(
        ppo_summary[f"{metric}_mean"]
    )
    random_value = float(
        random_summary[f"{metric}_mean"]
    )

    difference = ppo_value - random_value

    if np.isclose(random_value, 0.0):
        percentage = np.nan
    else:
        percentage = (
            difference / abs(random_value)
        ) * 100.0

    improvement_rows.append(
        {
            "metric": metric,
            "ppo_mean": ppo_value,
            "random_mean": random_value,
            "absolute_difference": difference,
            "ppo_improvement_percent": percentage,
        }
    )

improvement_df = pd.DataFrame(
    improvement_rows
)

display(comparison_df)
display(improvement_df)

In [ ]:
ppo_bankruptcy_rate = float(
    ppo_results["bankrupt"].mean()
)
random_bankruptcy_rate = float(
    random_results["bankrupt"].mean()
)

winner_reward = (
    "PPO"
    if ppo_results["total_reward"].mean()
    > random_results["total_reward"].mean()
    else "Random"
)

winner_savings = (
    "PPO"
    if ppo_results["final_savings"].mean()
    > random_results["final_savings"].mean()
    else "Random"
)

print("PERFORMANCE SUMMARY")
print("-" * 60)
print(
    "Average PPO reward:",
    round(ppo_results["total_reward"].mean(), 4),
)
print(
    "Average random reward:",
    round(random_results["total_reward"].mean(), 4),
)
print(
    "Average PPO final savings:",
    round(ppo_results["final_savings"].mean(), 2),
)
print(
    "Average random final savings:",
    round(random_results["final_savings"].mean(), 2),
)
print(
    "PPO bankruptcy rate:",
    f"{ppo_bankruptcy_rate:.2%}",
)
print(
    "Random bankruptcy rate:",
    f"{random_bankruptcy_rate:.2%}",
)
print("Reward winner:", winner_reward)
print("Savings winner:", winner_savings)

## 8. Generate comparison graphs

In [ ]:
def save_bar_chart(
    values,
    labels,
    title,
    ylabel,
    filename,
):
    plt.figure(figsize=(8, 5))
    plt.bar(labels, values)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xlabel("Policy")
    plt.tight_layout()
    plt.savefig(
        Path("graphs") / filename,
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()


save_bar_chart(
    [
        ppo_results["total_reward"].mean(),
        random_results["total_reward"].mean(),
    ],
    ["PPO", "Random"],
    "Average Total Reward",
    "Reward",
    "reward_comparison.png",
)

save_bar_chart(
    [
        ppo_results["final_savings"].mean(),
        random_results["final_savings"].mean(),
    ],
    ["PPO", "Random"],
    "Average Final Savings",
    "Savings",
    "savings_comparison.png",
)

save_bar_chart(
    [
        ppo_results["total_net_income"].mean(),
        random_results["total_net_income"].mean(),
    ],
    ["PPO", "Random"],
    "Average Total Net Income",
    "Net income",
    "net_income_comparison.png",
)

save_bar_chart(
    [
        ppo_results["years_survived"].mean(),
        random_results["years_survived"].mean(),
    ],
    ["PPO", "Random"],
    "Average Years Survived",
    "Years",
    "survival_comparison.png",
)

save_bar_chart(
    [
        ppo_bankruptcy_rate * 100.0,
        random_bankruptcy_rate * 100.0,
    ],
    ["PPO", "Random"],
    "Bankruptcy Rate",
    "Episodes (%)",
    "bankruptcy_comparison.png",
)

In [ ]:
plt.figure(figsize=(9, 5))
plt.boxplot(
    [
        ppo_results["total_reward"],
        random_results["total_reward"],
    ],
    tick_labels=["PPO", "Random"],
)
plt.title("Episode Reward Distribution")
plt.ylabel("Total reward")
plt.xlabel("Policy")
plt.tight_layout()
plt.savefig(
    "graphs/reward_distribution.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

In [ ]:
crop_frequency = (
    all_yearly.groupby(["policy", "crop"])
    .size()
    .unstack(fill_value=0)
)

crop_frequency_percentage = (
    crop_frequency.div(
        crop_frequency.sum(axis=1),
        axis=0,
    )
    * 100.0
)

crop_frequency_percentage.T.plot(
    kind="bar",
    figsize=(13, 6),
)
plt.title("Crop Selection Frequency by Policy")
plt.ylabel("Selections (%)")
plt.xlabel("Crop")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(
    "graphs/crop_frequency.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

display(crop_frequency_percentage)

In [ ]:
average_savings_by_year = (
    all_yearly.groupby(["policy", "year"])[
        "savings"
    ]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 6))

for policy_name, group in average_savings_by_year.groupby(
    "policy"
):
    plt.plot(
        group["year"],
        group["savings"],
        marker="o",
        label=policy_name,
    )

plt.title("Average Savings Across Planning Years")
plt.xlabel("Year")
plt.ylabel("Average savings")
plt.legend()
plt.tight_layout()
plt.savefig(
    "graphs/savings_over_time.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

## 9. Save evaluation results

In [ ]:
ppo_results.to_csv(
    "results/ppo_evaluation_results.csv",
    index=False,
)
random_results.to_csv(
    "results/random_evaluation_results.csv",
    index=False,
)
all_results.to_csv(
    "results/all_episode_results.csv",
    index=False,
)
ppo_yearly.to_csv(
    "results/ppo_yearly_decisions.csv",
    index=False,
)
random_yearly.to_csv(
    "results/random_yearly_decisions.csv",
    index=False,
)
comparison_df.to_csv(
    "results/ppo_vs_random_statistical_summary.csv",
    index=False,
)
improvement_df.to_csv(
    "results/ppo_vs_random_comparison.csv",
    index=False,
)
crop_frequency_percentage.to_csv(
    "results/crop_frequency_percent.csv"
)

report = {
    "evaluation_episodes_per_policy": N_EVAL_EPISODES,
    "horizon_years": HORIZON_YEARS,
    "initial_savings": INITIAL_SAVINGS,
    "model_used": str(MODEL_PATH),
    "reward_winner": winner_reward,
    "savings_winner": winner_savings,
    "ppo_bankruptcy_rate": ppo_bankruptcy_rate,
    "random_bankruptcy_rate": random_bankruptcy_rate,
    "ppo_average_reward": float(
        ppo_results["total_reward"].mean()
    ),
    "random_average_reward": float(
        random_results["total_reward"].mean()
    ),
    "ppo_average_final_savings": float(
        ppo_results["final_savings"].mean()
    ),
    "random_average_final_savings": float(
        random_results["final_savings"].mean()
    ),
}

Path(
    "results/performance_summary.json"
).write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

print("All CSV, JSON and graph files were saved.")

## 10. Download the complete Phase 6 results package

In [ ]:
from google.colab import files

OUTPUT_ZIP = "Farmer_RL_Phase6_Results.zip"

with zipfile.ZipFile(
    OUTPUT_ZIP,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    for folder in ["results", "graphs"]:
        for path in Path(folder).rglob("*"):
            if path.is_file():
                archive.write(
                    path,
                    path.as_posix(),
                )

print("Created:", OUTPUT_ZIP)
files.download(OUTPUT_ZIP)

## Phase 6 completion checklist

After running all cells, confirm that you have:

- `ppo_evaluation_results.csv`
- `random_evaluation_results.csv`
- `ppo_vs_random_comparison.csv`
- Reward and savings comparison graphs
- Crop-frequency graph
- `Farmer_RL_Phase6_Results.zip`

The next project phase is **Phase 7 — Results Visualisation and Interpretation**.